# Clase 1 — Actividad práctica: análisis exploratorio de datos
**Machine Learning · Especialización en Analítica de Datos**
Dr. Oscar Loyola Valenzuela

**Objetivo.** Aplicar las cuatro etapas del análisis exploratorio —estructura, calidad, análisis univariado y análisis bivariado/multivariado— y traducir los hallazgos en decisiones para el modelado.

**Conjunto de datos.** Palmer Penguins (Gorman et al., 2014; Horst et al., 2020): mediciones de 344 pingüinos de tres especies en el archipiélago Palmer, Antártida.

> Ejecute las celdas en orden. Las celdas marcadas como **Ejercicio** requieren completar código o responder por escrito.

## 0. Preparación del entorno
En Google Colab es necesario instalar el paquete que contiene los datos.

In [ ]:
%pip install -q palmerpenguins

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from palmerpenguins import load_penguins

COLORES = {"Adelie": "#00897B", "Chinstrap": "#D97706", "Gentoo": "#7C3AED"}
NUM = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

df = load_penguins()

## 1. Estructura

In [ ]:
print("Dimensiones (filas, columnas):", df.shape)
df.head()

In [ ]:
df.dtypes

### Ejercicio 1
Clasifique cada columna según la taxonomía vista en clase y responda: ¿cuál es la **unidad de observación** (qué representa una fila)?

| Columna | Tipo |
|---|---|
| `species` | |
| `island` | |
| `bill_length_mm` | |
| `bill_depth_mm` | |
| `flipper_length_mm` | |
| `body_mass_g` | |
| `sex` | |
| `year` | |

## 2. Calidad de los datos
### 2.1 Valores faltantes y duplicados

In [ ]:
resumen = pd.DataFrame({"faltantes": df.isna().sum(),
                        "porcentaje": (100 * df.isna().mean()).round(1)})
print("Filas duplicadas:", df.duplicated().sum())
resumen

In [ ]:
# Registros con al menos un valor faltante
df[df.isna().any(axis=1)]

### Ejercicio 2
1. ¿Qué patrón observa en los registros con faltantes? ¿Faltan las mismas columnas en todos?
2. Proponga un tratamiento para las mediciones morfológicas y otro para `sex`, y justifíquelos.

### 2.2 Valores atípicos: regla del rango intercuartílico

In [ ]:
def atipicos_ric(serie):
    """Devuelve una máscara booleana con los valores fuera de [Q1 - 1,5·RIC, Q3 + 1,5·RIC]."""
    q1, q3 = serie.quantile([0.25, 0.75])
    ric = q3 - q1
    return (serie < q1 - 1.5 * ric) | (serie > q3 + 1.5 * ric)

datos = df.dropna(subset=NUM)

# Aplicación global
print("Atípicos de masa corporal (conjunto completo):", atipicos_ric(datos["body_mass_g"]).sum())

### Ejercicio 3
Aplique la función `atipicos_ric` **por especie** a la masa corporal. Sugerencia: utilice `groupby("species")["body_mass_g"].transform(...)` o un ciclo sobre las especies. ¿Cuántos atípicos encuentra y en qué especie?

In [ ]:
# Complete aquí
mascara = datos.groupby("species")["body_mass_g"].transform(atipicos_ric)
datos.loc[mascara, ["species", "island", "body_mass_g", "sex"]]

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.boxplot([datos.loc[datos.species == s, "body_mass_g"] for s in COLORES], patch_artist=True)
ax.set_xticks([1, 2, 3], list(COLORES))
ax.set_ylabel("Masa corporal (g)")
plt.show()

## 3. Análisis univariado

In [ ]:
datos[NUM].describe().round(1)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.5))
for ax, col in zip(axes, NUM):
    ax.hist(datos[col], bins=22, color="#00897B", edgecolor="white")
    ax.axvline(datos[col].mean(), color="#D97706", label="Media")
    ax.axvline(datos[col].median(), color="black", ls="--", label="Mediana")
    ax.set_title(col)
axes[0].legend()
plt.tight_layout(); plt.show()

In [ ]:
df["species"].value_counts()

### Ejercicio 4
1. ¿Qué variable presenta una distribución bimodal?
2. Construya el histograma de esa variable **desagregado por especie** y explique el origen de la bimodalidad.

In [ ]:
# Complete aquí
fig, ax = plt.subplots(figsize=(7, 4))
for especie, color in COLORES.items():
    ax.hist(datos.loc[datos.species == especie, "flipper_length_mm"],
            bins=20, alpha=0.6, color=color, label=especie)
ax.set_xlabel("Largo de la aleta (mm)")
ax.set_ylabel("Frecuencia")
ax.legend(); plt.show()

## 4. Análisis bivariado y multivariado
### 4.1 Variables categóricas

In [ ]:
pd.crosstab(df["island"], df["species"], margins=True)

**Pregunta.** ¿Debería un modelo de identificación de especies utilizar la isla como variable explicativa? Argumente considerando el uso previsto del modelo.

### 4.2 Correlación entre variables numéricas

In [ ]:
datos[NUM].corr().round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(datos[NUM].corr(), cmap="PiYG", vmin=-1, vmax=1)
ax.set_xticks(range(4), NUM, rotation=30, ha="right"); ax.set_yticks(range(4), NUM)
fig.colorbar(im); plt.show()

### Ejercicio 5 — La paradoja de Simpson
Calcule la correlación entre `bill_length_mm` y `bill_depth_mm`:
1. con todos los registros;
2. dentro de cada especie.

Compare los signos y explique el resultado.

In [ ]:
# Complete aquí
r_global = datos["bill_length_mm"].corr(datos["bill_depth_mm"])
print(f"Correlación global: {r_global:.2f}")
for especie, g in datos.groupby("species"):
    print(f"{especie:10s}: {g['bill_length_mm'].corr(g['bill_depth_mm']):.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
axes[0].scatter(datos["bill_length_mm"], datos["bill_depth_mm"], s=15, color="gray")
axes[0].set_title("Todos los registros")
for especie, color in COLORES.items():
    g = datos[datos.species == especie]
    axes[1].scatter(g["bill_length_mm"], g["bill_depth_mm"], s=15, color=color, label=especie)
axes[1].set_title("Por especie"); axes[1].legend()
for ax in axes:
    ax.set_xlabel("Largo del pico (mm)")
axes[0].set_ylabel("Profundidad del pico (mm)")
plt.show()

## 5. Del análisis exploratorio al modelo
Complete la tabla con al menos cuatro hallazgos propios y la decisión de modelado que cada uno implica.

| Hallazgo | Decisión para el modelado |
|---|---|
| | |
| | |
| | |
| | |

**Pregunta final.** Proponga un ejemplo de su ámbito profesional en que analizar datos agregados, sin desagregar por grupo, conduzca a una decisión errónea.

---
**Referencias**
- Gorman, K. B., Williams, T. D., & Fraser, W. R. (2014). Ecological sexual dimorphism and environmental variability within a community of Antarctic penguins (genus *Pygoscelis*). *PLoS ONE*, 9(3), e90081.
- Horst, A. M., Hill, A. P., & Gorman, K. B. (2020). *palmerpenguins: Palmer Archipelago (Antarctica) penguin data*.
- McKinney, W. (2022). *Python for Data Analysis* (3.ª ed.). O'Reilly.
- Tukey, J. W. (1977). *Exploratory Data Analysis*. Addison-Wesley.